# Baseline eval on Kaggle — `post-training-lab`

Thin launcher. Clones the repo, installs the `[eval]` extra, runs `make eval-baseline`,
produces `results/metrics.json` with the un-tuned Qwen2.5-0.5B-Instruct row.

Sister of `notebooks/colab/run_eval_baseline_colab.ipynb` — same flow, Kaggle-flavored.

**Before you start**
1. Right sidebar → **Settings**:
   - **Accelerator** → `GPU T4 x2` (free, generous) or `GPU P100`.
   - **Internet** → `On`. _Required for `git clone` and HF model download._ First-time use needs a one-off phone verification on your Kaggle account.
   - **Persistence** → `Variables and Files` (so `/kaggle/working` survives across re-runs in this session).
2. Top bar → **Add-ons → Secrets**, add:
   - `HF_TOKEN` — your Hugging Face read token (`huggingface.co/settings/tokens`).
   - `WANDB_API_KEY` — optional, only if you want runs to log to W&B.
3. Update `GITHUB_USER` in the clone cell below to your handle.

Reference: `PROJECT.md` §3 (Colab/Kaggle bootstrap), §4.3 (HF cache), §5.4 (smoke-test discipline).

## 1. GPU sanity check

If `nvidia-smi` fails or shows no GPU, the accelerator isn't selected — fix it in Settings (right sidebar) before continuing.

In [ ]:
!nvidia-smi

## 2. Clone the repo

Kaggle's writable scratch dir is `/kaggle/working` (Colab uses `/content`). The clone is
idempotent — re-running this cell wipes and re-clones, so a partial earlier attempt
never poisons the next run. If the repo is private, generate a fine-grained PAT and
use: `https://<TOKEN>@github.com/<user>/post-training-lab.git`.

In [ ]:
GITHUB_USER = "agaonker"
REPO        = "post-training-lab"
BRANCH      = "main"
REPO_PATH   = f"/kaggle/working/{REPO}"   # absolute — idempotent across re-runs

import os, shutil
os.chdir("/kaggle/working")
if os.path.exists(REPO_PATH):
    shutil.rmtree(REPO_PATH)
!git clone --depth 1 --branch {BRANCH} https://github.com/{GITHUB_USER}/{REPO}.git {REPO_PATH}
os.chdir(REPO_PATH)
!git log -1 --oneline

## 3. Point HF cache at the working dir

Kaggle has no Google Drive. Two levels of persistence are available:

- **In-session** (this cell): `/kaggle/working/hf-cache` survives re-runs of cells within the
  current session. Cheap, no setup.
- **Cross-session**: after the first run, **File → Save Version → Save & Run All**. The
  output snapshot can be attached as an input dataset on a subsequent notebook (read-only
  is fine for HF) — set `HF_HOME=/kaggle/input/<dataset-name>/hf-cache` instead. This is
  the Kaggle equivalent of mounting Drive on Colab.

In [ ]:
os.environ["HF_HOME"] = "/kaggle/working/hf-cache"
os.makedirs(os.environ["HF_HOME"], exist_ok=True)
print("HF_HOME =", os.environ["HF_HOME"])

## 4. Install the project

Kaggle doesn't ship `uv`, so we fall back to plain `pip install -e .[eval]` — same
bootstrap as the Colab notebook. This pulls in `lm-eval`, `transformers`, `trl`,
`peft`, etc. First install on a fresh kernel is 2–5 minutes.

In [ ]:
!pip install -q -e .[eval]

## 5. Inject secrets

Reads from Kaggle's Secrets panel (Add-ons → Secrets). Nothing is printed.
Kaggle's API is `kaggle_secrets.UserSecretsClient` — different from Colab's `google.colab.userdata`.

In [ ]:
from kaggle_secrets import UserSecretsClient
us = UserSecretsClient()

for key in ("HF_TOKEN", "WANDB_API_KEY"):
    try:
        os.environ[key] = us.get_secret(key)
        print(f"{key:14s} loaded")
    except Exception:
        print(f"{key:14s} not set (optional)")

# huggingface_hub picks up HF_TOKEN automatically when present.
# wandb mode -> disabled if no API key, to avoid an interactive prompt mid-eval.
if not os.environ.get("WANDB_API_KEY"):
    os.environ["WANDB_MODE"] = "disabled"

## 6. Patch dtype to fp16 on T4 / P100

Neither T4 (Turing) nor P100 (Pascal) has native bf16 — both emulate it in software
and run roughly half-speed. Free Kaggle gives one of these by default, so this cell
almost always fires. We swap the resolved config's dtype before launching the eval.
Skip the patch only if you're on a Volta/Ampere/Hopper GPU (V100/A100/H100/L4).

In [ ]:
import subprocess, re, pathlib
gpu_name = subprocess.check_output(
    ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"]
).decode().strip()
print("GPU:", gpu_name)

if "T4" in gpu_name or "P100" in gpu_name:
    # Override on base.yaml — baseline.yaml inherits from it.
    p = pathlib.Path("configs/base.yaml")
    p.write_text(re.sub(r"dtype:\s*bfloat16", "dtype: float16", p.read_text()))
    print("Patched configs/base.yaml -> dtype: float16 (native fp16 beats emulated bf16).")
else:
    print("Native bf16 GPU — leaving config untouched.")

## 7. Smoke test (10 samples per task)

Run **before** the full eval. If anything is broken — bad dataset args, OOM, tokenizer
mismatch — it fails here in 2–3 minutes instead of an hour into IFEval. PROJECT.md §5.4:
*"Never launch a paid GPU on code you haven't run 50 steps of on free tier."*

In [ ]:
# The Makefile auto-detects whether `uv` is on PATH AND `.venv` is in cwd.
# Kaggle has neither, so RUNNER falls back to empty and we use the kernel's
# system Python directly — the one `pip install -e .[eval]` above populated.
!make eval-smoke

## 8. Full baseline eval

All four tasks at their configured shots/limits (see `configs/base.yaml`):

| Task | Few-shot | Limit |
|---|---|---|
| MMLU | 5 | 1000 |
| GSM8K | 8 | full |
| TruthfulQA MC2 | 0 | full |
| IFEval | 0 | full |

Expected wall-clock on a free T4: **~30–60 min** total, IFEval dominates. Well
under Kaggle's 9-hour session cap.

In [ ]:
!make eval-baseline

## 9. Inspect the results

In [ ]:
import json, pathlib
doc = json.loads(pathlib.Path("results/metrics.json").read_text())
print(json.dumps(doc, indent=2))

## 10. Get `metrics.json` back to your laptop

Kaggle has no `google.colab.files.download` equivalent. Two options:

- **Quick (one-off)**: right sidebar → **Output** tab → expand `working/post-training-lab/results/` → click `metrics.json` → ⋯ menu → **Download**. Drop into your local `results/` and commit.
- **Snapshot (recommended for the canonical row)**: top bar → **File → Save Version → Save & Run All**. The committed version archives `/kaggle/working/` as an immutable output you can re-attach, share, or wget from your laptop via the Kaggle API:
  ```
  kaggle kernels output <your-username>/<notebook-slug> -p ./results/
  ```